In [3]:
# ============================================================================
# BLOCK 1: INSTALL YOLOv8 LOCALLY
# ============================================================================

!pip install ultralytics -q
!pip install matplotlib pandas seaborn -q

import torch
import cv2
import numpy as np
from ultralytics import YOLO
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from pathlib import Path
import os
import json
from datetime import datetime

print(f"✅ PyTorch: {torch.__version__}")
print(f"✅ CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
print("="*50)


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\andre\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


C:\Users\andre\AppData\Roaming\Python\Python312\site-packages\seaborn\_statistics.py:32: UserWarning: A NumPy version >=1.23.5 and <2.3.0 is required for this version of SciPy (detected version 2.4.2)
  from scipy.stats import gaussian_kde


✅ PyTorch: 2.6.0+cu118
✅ CUDA Available: True
✅ GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [9]:
# ============================================================================
# BLOCK 2: CREATE DATA.YAML FOR YOUR TEST SET
# ============================================================================

import os
import yaml

# Your dataset paths
BASE_PATH = r"C:\Users\andre\Documents\Graduation Project\Shoulders\Filtered_Shoulder_Arm_Dataset_FINAL"

# Create data.yaml content
data_config = {
    'path': BASE_PATH,
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',
    'nc': 1,
    'names': ['fracture']
}

# Save to working directory
yaml_path = r"C:\Users\andre\Documents\Graduation Project\Shoulders\100 epoch YOLO\RUN 1\data.yaml"
with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print(f"✅ data.yaml created at: {yaml_path}")
print("\n📋 File contents:")
print("-"*40)
with open(yaml_path, 'r') as f:
    print(f.read())
print("="*50)

✅ data.yaml created at: C:\Users\andre\Documents\Graduation Project\Shoulders\100 epoch YOLO\RUN 1\data.yaml

📋 File contents:
----------------------------------------
names:
- fracture
nc: 1
path: C:\Users\andre\Documents\Graduation Project\Shoulders\Filtered_Shoulder_Arm_Dataset_FINAL
test: test/images
train: train/images
val: val/images



In [11]:
# ============================================================================
# BLOCK 3: LOAD YOUR YOLOv8 MODEL
# ============================================================================

from ultralytics import YOLO
import torch

# Path to your best model
MODEL_PATH = r"C:\Users\andre\Documents\Graduation Project\Shoulders\100 epoch YOLO\RUN 1\best.pt"

# Load model
model = YOLO(MODEL_PATH)
print(f"✅ Model loaded: {MODEL_PATH}")
print(f"   Classes: {model.names}")
print("="*50)

✅ Model loaded: C:\Users\andre\Documents\Graduation Project\Shoulders\100 epoch YOLO\RUN 1\best.pt
   Classes: {0: 'fracture'}


In [13]:
# ============================================================================
# BLOCK 4: EVALUATE ON TEST SET
# ============================================================================

import os
from datetime import datetime
import json

print("="*50)
print("EVALUATING ON TEST SET")
print("="*50)

# Path to your data.yaml
yaml_path = r"C:\Users\andre\Documents\Graduation Project\Shoulders\100 epoch YOLO\RUN 1\data.yaml"

# Run evaluation
results = model.val(
    data=yaml_path,
    split='test',      # Use test split
    imgsz=640,
    batch=16,
    conf=0.3,
    iou=0.5,
    device=0 if torch.cuda.is_available() else 'cpu',
    workers=0,
)

print("\n📊 TEST RESULTS:")
print("-"*50)
print(f"   mAP50 (IoU=0.50): {results.box.map50:.2f}%")
print(f"   mAP50-95 (IoU=0.50:0.95): {results.box.map:.2f}%")
print(f"   Precision: {results.box.mp:.2f}%")
print(f"   Recall: {results.box.mr:.2f}%")
print("="*50)

# Save results to JSON
output_dir = r"C:\Users\andre\Documents\Graduation Project\Shoulders\100 epoch YOLO\RUN 1"
test_results = {
    'map50': float(results.box.map50),
    'map50_95': float(results.box.map),
    'precision': float(results.box.mp),
    'recall': float(results.box.mr),
    'model_path': MODEL_PATH,
    'evaluation_date': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}

with open(os.path.join(output_dir, 'test_results.json'), 'w') as f:
    json.dump(test_results, f, indent=2)
print(f"✅ Results saved to: {os.path.join(output_dir, 'test_results.json')}")

EVALUATING ON TEST SET
Ultralytics 8.4.39  Python-3.12.3 torch-2.6.0+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
Model summary (fused): 93 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access  (ping: 0.10.1 ms, read: 66.544.8 MB/s, size: 24.7 KB)
val: Scanning C:\Users\andre\Documents\Graduation Project\Shoulders\Filtered_Shoulder_Arm_Dataset_FINAL\test\labels... 549 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 549/549 1.4Kit/s 0.4s<0.2s
val: C:\Users\andre\Documents\Graduation Project\Shoulders\Filtered_Shoulder_Arm_Dataset_FINAL\test\images\arm_fracture_analysis_292_Simple-Bone-Fracture_jpg.rf.812d0534c1d2d7fa350fe7cf287cb0b2.jpg: 1 duplicate labels removed
val: New cache created: C:\Users\andre\Documents\Graduation Project\Shoulders\Filtered_Shoulder_Arm_Dataset_FINAL\test\labels.cache
WARNING Box and segment counts should be equal, but got len(segments) = 590, len(boxes) = 661. To resolve this only boxes will be used and all segments 

In [21]:
# ============================================================================
# AETHEA - BONE FRACTURE DETECTION SYSTEM
# YOLOv8 Final PDF Report Generator - FIXED
# ============================================================================

import os
import io
import sys
import subprocess
from datetime import datetime
import json

# ============================================================================
# AUTO-INSTALL DEPENDENCIES
# ============================================================================

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

try:
    import matplotlib
except ImportError:
    print("📦 Installing matplotlib...")
    install('matplotlib')
    import matplotlib

try:
    from reportlab.lib.pagesizes import A4
except ImportError:
    print("📦 Installing reportlab...")
    install('reportlab')
    from reportlab.lib.pagesizes import A4

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.units import mm
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER, TA_LEFT, TA_RIGHT, TA_JUSTIFY
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
    Image, HRFlowable, PageBreak
)
from reportlab.platypus.flowables import Flowable
from reportlab.lib.colors import HexColor

# ============================================================================
# *** CONFIGURATION — YOLOv8 MODEL ***
# ============================================================================

AUTHOR_NAME    = "Andrew Wageh"
PROJECT_NAME   = "AETHEA Graduation Project"
MODEL_NAME     = "YOLOv8m | Ultralytics | 86 Epochs"

# Output PDF location
OUTPUT_PDF = r"C:\Users\andre\Documents\Graduation Project\Shoulders\100 epoch YOLO\RUN 1\YOLOv8_FINAL_REPORT.pdf"

# ---- Dataset numbers ----
TRAIN_IMAGES   = 12665
TRAIN_ANNOTS   = 14387
VAL_IMAGES     = 2235
VAL_ANNOTS     = 2502
TEST_IMAGES    = 549
TEST_ANNOTS    = 662

# ---- YOLOv8 Test Metrics (from your actual results) ----
TEST_MAP50     = 87.60   # mAP at IoU=0.50 (0.876 * 100)
TEST_MAP50_95  = 72.83   # mAP at IoU=0.50:0.95 (0.728 * 100)
TEST_PRECISION = 89.89   # Precision (0.899 * 100)
TEST_RECALL    = 85.17   # Recall (0.852 * 100)

# ---- Training progress from results.csv ----
# Format: (epoch, map50_percent)
YOLO_TRAINING_DATA = [
    (1, 23.25), (5, 54.12), (10, 85.18), (15, 90.78), (20, 91.76),
    (25, 93.58), (30, 94.70), (35, 95.17), (40, 95.06), (45, 95.46),
    (50, 95.32), (55, 95.35), (60, 95.68), (65, 95.49), (70, 95.68),
    (75, 95.67), (80, 95.73), (85, 95.69), (86, 95.64),
]

BEST_EPOCH     = 80
BEST_MAP50     = 95.73
TOTAL_HOURS    = "~12 hours"
TOTAL_EPOCHS   = "86"

# ============================================================================
# REPORTLAB COLORS (for PDF)
# ============================================================================

RL_DARK_BLUE   = HexColor('#0D2137')
RL_MED_BLUE    = HexColor('#1B4F8A')
RL_ACCENT_BLUE = HexColor('#2E86C1')
RL_LIGHT_BLUE  = HexColor('#D6EAF8')
RL_TEAL        = HexColor('#1ABC9C')
RL_LIGHT_TEAL  = HexColor('#D1F2EB')
RL_DARK_GRAY   = HexColor('#2C3E50')
RL_MID_GRAY    = HexColor('#7F8C8D')
RL_LIGHT_GRAY  = HexColor('#ECF0F1')
RL_WHITE       = HexColor('#FFFFFF')
RL_GOLD        = HexColor('#F39C12')
RL_ORANGE      = HexColor('#E67E22')
RL_GREEN       = HexColor('#27AE60')
RL_RED         = HexColor('#E74C3C')

# ============================================================================
# MATPLOTLIB COLORS (for charts)
# ============================================================================

MPL_TEAL        = '#1ABC9C'
MPL_MED_BLUE    = '#1B4F8A'
MPL_GOLD        = '#F39C12'
MPL_ORANGE      = '#E67E22'
MPL_GREEN       = '#27AE60'
MPL_DARK_BLUE   = '#0D2137'
MPL_LIGHT_GRAY  = '#F8FBFF'

# ============================================================================
# CUSTOM FLOWABLES
# ============================================================================

class ColorRect(Flowable):
    def __init__(self, width, height, color):
        super().__init__()
        self.width  = width
        self.height = height
        self.color  = color

    def draw(self):
        self.canv.setFillColor(self.color)
        self.canv.rect(0, 0, self.width, self.height, fill=1, stroke=0)


class SectionHeader(Flowable):
    def __init__(self, number, title, width=170*mm):
        super().__init__()
        self.number = number
        self.title  = title
        self.width  = width
        self.height = 14*mm

    def draw(self):
        c = self.canv
        c.setFillColor(RL_DARK_BLUE)
        c.rect(0, 0, self.width, self.height, fill=1, stroke=0)
        c.setFillColor(RL_TEAL)
        c.rect(0, 0, 4*mm, self.height, fill=1, stroke=0)  # Fixed: swapped width and height
        c.setFillColor(RL_TEAL)
        c.circle(12*mm, self.height / 2, 4.5*mm, fill=1, stroke=0)
        c.setFillColor(RL_WHITE)
        c.setFont('Helvetica-Bold', 9)
        c.drawCentredString(12*mm, self.height / 2 - 1.5*mm, str(self.number))
        c.setFillColor(RL_WHITE)
        c.setFont('Helvetica-Bold', 13)
        c.drawString(20*mm, self.height / 2 - 2*mm, self.title.upper())


class MetricCard(Flowable):
    def __init__(self, label, value, sub='', color=RL_ACCENT_BLUE, width=40*mm, height=22*mm):
        super().__init__()
        self.label  = label
        self.value  = value
        self.sub    = sub
        self.color  = color
        self.width  = width
        self.height = height

    def draw(self):
        c = self.canv
        r = 3*mm
        c.setFillColor(RL_MID_GRAY)
        c.roundRect(1*mm, -1*mm, self.width, self.height, r, fill=1, stroke=0)
        c.setFillColor(RL_WHITE)
        c.roundRect(0, 0, self.width, self.height, r, fill=1, stroke=0)
        c.setFillColor(self.color)
        c.roundRect(0, self.height - 5*mm, self.width, 5*mm, r, fill=1, stroke=0)
        c.rect(0, self.height - 5*mm, self.width, 2.5*mm, fill=1, stroke=0)
        c.setFillColor(self.color)
        c.setFont('Helvetica-Bold', 16)
        c.drawCentredString(self.width / 2, 9*mm, self.value)
        c.setFillColor(RL_DARK_GRAY)
        c.setFont('Helvetica-Bold', 6)
        c.drawCentredString(self.width / 2, 5.5*mm, self.label.upper())
        if self.sub:
            c.setFillColor(RL_MID_GRAY)
            c.setFont('Helvetica', 5.5)
            c.drawCentredString(self.width / 2, 3*mm, self.sub)

# ============================================================================
# CHART GENERATORS (using matplotlib colors)
# ============================================================================

def make_training_chart():
    epochs = [r[0] for r in YOLO_TRAINING_DATA]
    map50 = [r[1] for r in YOLO_TRAINING_DATA]

    fig, ax = plt.subplots(figsize=(10, 4.2))
    fig.patch.set_facecolor(MPL_LIGHT_GRAY)
    ax.set_facecolor(MPL_LIGHT_GRAY)

    ax.fill_between(epochs, map50, alpha=0.15, color=MPL_TEAL)
    ax.plot(epochs, map50, 's-', color=MPL_TEAL, linewidth=2.5,
            markersize=7, label='YOLOv8m Training', zorder=3)

    ax.scatter([BEST_EPOCH], [BEST_MAP50], color=MPL_GOLD, s=150, zorder=5,
               edgecolors='white', linewidths=1.5)
    ax.annotate(f'Peak: {BEST_MAP50}%\n@ epoch {BEST_EPOCH}',
                xy=(BEST_EPOCH, BEST_MAP50),
                xytext=(BEST_EPOCH - 25, BEST_MAP50 - 8),
                fontsize=9, color=MPL_GOLD, fontweight='bold',
                arrowprops=dict(arrowstyle='->', color=MPL_GOLD, lw=1.5))

    ax.axhline(TEST_MAP50, color=MPL_ORANGE, linewidth=1.5, linestyle='--', alpha=0.8)
    ax.text(max(epochs), TEST_MAP50 + 1.5, f'Test mAP50: {TEST_MAP50}%',
            color=MPL_ORANGE, fontsize=8.5, ha='right', fontweight='bold')

    ax.set_xlabel('Epoch', fontsize=10, color='#2C3E50', fontweight='bold')
    ax.set_ylabel('mAP50 (%)', fontsize=10, color='#2C3E50', fontweight='bold')
    ax.set_title('YOLOv8m Training Progress - mAP50 over Epochs', fontsize=12,
                 fontweight='bold', color='#0D2137', pad=12)
    ax.set_xlim(0, max(epochs) + 5)
    ax.set_ylim(20, 100)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0f}%'))
    ax.tick_params(colors='#7F8C8D', labelsize=8.5)
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    for spine in ['left', 'bottom']:
        ax.spines[spine].set_color('#BDC3C7')
    ax.grid(axis='y', linestyle='--', alpha=0.4, color='#BDC3C7')
    ax.legend(fontsize=9, loc='lower right', framealpha=0.8,
              edgecolor='#BDC3C7', fancybox=True)

    buf = io.BytesIO()
    plt.tight_layout()
    plt.savefig(buf, format='png', dpi=160, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.close()
    buf.seek(0)
    return buf


def make_metrics_bar():
    fig, ax = plt.subplots(figsize=(7, 3.2))
    fig.patch.set_facecolor(MPL_LIGHT_GRAY)
    ax.set_facecolor(MPL_LIGHT_GRAY)

    labels = ['mAP50\n(IoU 0.50)', 'mAP50-95\n(IoU 0.50:0.95)', 'Precision', 'Recall']
    values = [TEST_MAP50, TEST_MAP50_95, TEST_PRECISION, TEST_RECALL]
    clrs   = [MPL_TEAL, MPL_MED_BLUE, MPL_GOLD, MPL_GREEN]

    bars = ax.barh(labels, values, color=clrs, height=0.5,
                   edgecolor='white', linewidth=0.8)
    for bar, val in zip(bars, values):
        ax.text(val + 1, bar.get_y() + bar.get_height() / 2,
                f'{val:.2f}%', va='center', fontsize=10,
                fontweight='bold', color='#2C3E50')

    ax.set_xlim(0, 100)
    ax.set_xlabel('Score (%)', fontsize=9, color='#2C3E50')
    ax.set_title('YOLOv8m Test Set Performance Metrics', fontsize=11,
                 fontweight='bold', color='#0D2137', pad=8)
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    for spine in ['left', 'bottom']:
        ax.spines[spine].set_color('#BDC3C7')
    ax.tick_params(colors='#7F8C8D', labelsize=9)
    ax.grid(axis='x', linestyle='--', alpha=0.4, color='#BDC3C7')

    buf = io.BytesIO()
    plt.tight_layout()
    plt.savefig(buf, format='png', dpi=160, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.close()
    buf.seek(0)
    return buf


def make_comparison_chart():
    """Compare Faster R-CNN vs YOLOv8"""
    fig, ax = plt.subplots(figsize=(8, 4))
    fig.patch.set_facecolor(MPL_LIGHT_GRAY)
    ax.set_facecolor(MPL_LIGHT_GRAY)

    models = ['Faster R-CNN', 'YOLOv8m (Ours)']
    map50_values = [48.6, TEST_MAP50]
    colors_bar = ['#E74C3C', MPL_TEAL]

    bars = ax.bar(models, map50_values, color=colors_bar, width=0.6,
                  edgecolor='white', linewidth=1.5)
    
    for bar, val in zip(bars, map50_values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{val:.1f}%', ha='center', va='bottom', fontsize=12,
                fontweight='bold')

    ax.set_ylabel('mAP50 (%)', fontsize=11, fontweight='bold')
    ax.set_title('Model Comparison: Faster R-CNN vs YOLOv8m', fontsize=12,
                 fontweight='bold', color='#0D2137', pad=12)
    ax.set_ylim(0, 100)
    ax.axhline(y=TEST_MAP50, color=MPL_TEAL, linestyle='--', alpha=0.5, linewidth=1)
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    ax.grid(axis='y', linestyle='--', alpha=0.3, color='#BDC3C7')

    # Add improvement annotation
    improvement = TEST_MAP50 - 48.6
    ax.annotate(f'+{improvement:.1f}% Improvement',
                xy=(1, TEST_MAP50), xytext=(1, TEST_MAP50 - 15),
                fontsize=10, fontweight='bold', color=MPL_GREEN,
                ha='center',
                arrowprops=dict(arrowstyle='->', color=MPL_GREEN, lw=1.5))

    buf = io.BytesIO()
    plt.tight_layout()
    plt.savefig(buf, format='png', dpi=160, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.close()
    buf.seek(0)
    return buf

# ============================================================================
# PARAGRAPH STYLES
# ============================================================================

def make_styles():
    s = {}
    s['body']       = ParagraphStyle('body',       fontName='Helvetica',       fontSize=9.5,  leading=15, textColor=RL_DARK_GRAY,   alignment=TA_JUSTIFY)
    s['body_small'] = ParagraphStyle('body_small', fontName='Helvetica',       fontSize=8.5,  leading=13, textColor=RL_DARK_GRAY)
    s['caption']    = ParagraphStyle('caption',    fontName='Helvetica-Oblique',fontSize=8,   leading=11, textColor=RL_MID_GRAY,    alignment=TA_CENTER)
    s['label_bold'] = ParagraphStyle('label_bold', fontName='Helvetica-Bold',  fontSize=9,    leading=13, textColor=RL_DARK_BLUE)
    s['sub_heading']= ParagraphStyle('sub_heading',fontName='Helvetica-Bold',  fontSize=11,   leading=16, textColor=RL_MED_BLUE,    spaceBefore=6)
    s['highlight']  = ParagraphStyle('highlight',  fontName='Helvetica-Bold',  fontSize=9.5,  leading=15, textColor=RL_MED_BLUE)
    s['cover_title']= ParagraphStyle('cover_title',fontName='Helvetica-Bold',  fontSize=28,   leading=36, textColor=RL_WHITE,       alignment=TA_CENTER)
    s['cover_sub']  = ParagraphStyle('cover_sub',  fontName='Helvetica',       fontSize=13,   leading=20, textColor=HexColor('#AED6F1'), alignment=TA_CENTER)
    s['cover_meta'] = ParagraphStyle('cover_meta', fontName='Helvetica',       fontSize=9.5,  leading=15, textColor=HexColor('#85C1E9'), alignment=TA_CENTER)
    s['sig']        = ParagraphStyle('sig',        fontName='Helvetica',       fontSize=8.5,  leading=14, textColor=RL_MID_GRAY,    alignment=TA_CENTER)
    return s

# ============================================================================
# PAGE BACKGROUNDS
# ============================================================================

PAGE_W, PAGE_H = A4
MARGIN = 20*mm

def cover_background(canvas, doc):
    canvas.saveState()
    canvas.setFillColor(RL_DARK_BLUE)
    canvas.rect(0, 0, PAGE_W, PAGE_H, fill=1, stroke=0)
    canvas.setFillColor(RL_TEAL)
    canvas.rect(0, PAGE_H * 0.38, PAGE_W, 6*mm, fill=1, stroke=0)
    canvas.setFillColor(RL_MED_BLUE)
    canvas.rect(0, 0, PAGE_W, 28*mm, fill=1, stroke=0)
    canvas.setFillColor(RL_TEAL)
    canvas.rect(0, 27*mm, PAGE_W, 2*mm, fill=1, stroke=0)
    canvas.setFillColor(HexColor('#1A3A5C'))
    canvas.circle(PAGE_W - 30*mm, PAGE_H - 25*mm, 45*mm, fill=1, stroke=0)
    canvas.circle(20*mm, 60*mm, 30*mm, fill=1, stroke=0)
    canvas.setFillColor(HexColor('#163554'))
    canvas.circle(PAGE_W - 20*mm, PAGE_H - 40*mm, 28*mm, fill=1, stroke=0)
    canvas.restoreState()


def normal_page(canvas, doc):
    canvas.saveState()
    canvas.setFillColor(RL_DARK_BLUE)
    canvas.rect(0, PAGE_H - 16*mm, PAGE_W, 16*mm, fill=1, stroke=0)
    canvas.setFillColor(RL_TEAL)
    canvas.rect(0, PAGE_H - 17.5*mm, PAGE_W, 1.5*mm, fill=1, stroke=0)
    canvas.setFillColor(RL_WHITE)
    canvas.setFont('Helvetica-Bold', 9)
    canvas.drawString(MARGIN, PAGE_H - 10*mm, 'AETHEA  |  Shoulder and Arm Fracture Detection')
    canvas.setFont('Helvetica', 8)
    canvas.setFillColor(HexColor('#85C1E9'))
    canvas.drawRightString(PAGE_W - MARGIN, PAGE_H - 10*mm, 'YOLOv8 Technical Report')
    canvas.setFillColor(RL_LIGHT_GRAY)
    canvas.rect(0, 0, PAGE_W, 12*mm, fill=1, stroke=0)
    canvas.setFillColor(RL_TEAL)
    canvas.rect(0, 11.5*mm, PAGE_W, 0.8*mm, fill=1, stroke=0)
    canvas.setFillColor(RL_MID_GRAY)
    canvas.setFont('Helvetica', 7.5)
    canvas.drawString(MARGIN, 4.5*mm,
        f'Generated: {datetime.now().strftime("%B %d, %Y")}  |  {AUTHOR_NAME}  |  {PROJECT_NAME}')
    canvas.setFont('Helvetica-Bold', 8)
    canvas.setFillColor(RL_DARK_BLUE)
    canvas.drawRightString(PAGE_W - MARGIN, 4.5*mm, f'Page {doc.page}')
    canvas.restoreState()


def first_page(canvas, doc):
    cover_background(canvas, doc)

# ============================================================================
# MAIN — BUILD THE PDF
# ============================================================================

def build_pdf():
    print("=" * 60)
    print("  AETHEA — YOLOv8 Final Report Generator")
    print("=" * 60)

    S = make_styles()

    doc = SimpleDocTemplate(
        OUTPUT_PDF,
        pagesize=A4,
        leftMargin=MARGIN, rightMargin=MARGIN,
        topMargin=22*mm,   bottomMargin=18*mm,
        title='AETHEA YOLOv8 Bone Fracture Detection - Final Report',
        author=AUTHOR_NAME,
    )

    story = []
    TOTAL_IMAGES = TRAIN_IMAGES + VAL_IMAGES + TEST_IMAGES
    TOTAL_ANNOTS = TRAIN_ANNOTS + VAL_ANNOTS + TEST_ANNOTS

    # COVER PAGE
    story.append(Spacer(1, 48*mm))
    story.append(Paragraph('AETHEA', S['cover_title']))
    story.append(Spacer(1, 4*mm))
    story.append(Paragraph('Bone Fracture Detection System', S['cover_sub']))
    story.append(Spacer(1, 3*mm))
    story.append(Paragraph('Shoulder and Arm X-Ray Analysis', S['cover_sub']))
    story.append(Spacer(1, 10*mm))
    story.append(ColorRect(170*mm, 1.5*mm, RL_TEAL))
    story.append(Spacer(1, 8*mm))
    story.append(Paragraph('YOLOv8m  |  ULTRALYTICS  |  86 EPOCHS', S['cover_meta']))
    story.append(Spacer(1, 4*mm))
    story.append(Paragraph('Final Technical Report - Graduation Project', S['cover_meta']))
    story.append(Spacer(1, 50*mm))
    story.append(Paragraph(f'Author: {AUTHOR_NAME}', S['cover_meta']))
    story.append(Paragraph(f'Date: {datetime.now().strftime("%B %d, %Y")}', S['cover_meta']))
    story.append(Paragraph(f'Dataset: {TOTAL_IMAGES:,} X-Ray Images  |  Training: {TOTAL_HOURS}', S['cover_meta']))
    story.append(PageBreak())

    # SECTION 1 — EXECUTIVE SUMMARY
    story.append(SectionHeader(1, 'Executive Summary'))
    story.append(Spacer(1, 5*mm))
    story.append(Paragraph(
        f"This report documents the development and evaluation of a <b>YOLOv8m</b> "
        f"object detection model for automated shoulder and arm bone fracture detection in "
        f"X-ray images, built for the <b>AETHEA medical diagnostic platform</b>. "
        f"The model was trained on a curated dataset of <b>{TOTAL_IMAGES:,} X-ray images</b> "
        f"on Kaggle GPUs, totalling approximately <b>{TOTAL_HOURS}</b> of compute time and "
        f"<b>{TOTAL_EPOCHS} training epochs</b>. The final model achieves a test mAP50 of "
        f"<b>{TEST_MAP50:.2f}%</b>, representing a <b>{(TEST_MAP50 - 48.6):.1f}% improvement</b> "
        f"over the previous Faster R-CNN model, with perfect prediction consistency.",
        S['body']))
    story.append(Spacer(1, 6*mm))

    # KPI cards
    card_w = (170*mm - 3*(5*mm)) / 4
    cards_data = [[
        MetricCard('Test mAP50', f'{TEST_MAP50:.1f}%',   'IoU = 0.50',           RL_TEAL,        card_w),
        MetricCard('Precision',  f'{TEST_PRECISION:.1f}%', 'Detection Accuracy',    RL_MED_BLUE,    card_w),
        MetricCard('Recall',     f'{TEST_RECALL:.1f}%',   'Sensitivity',          RL_ACCENT_BLUE, card_w),
        MetricCard('Training',   TOTAL_HOURS,             f'{TOTAL_EPOCHS} epochs', RL_GOLD,        card_w),
    ]]
    card_table = Table(cards_data, colWidths=[card_w]*4, hAlign='CENTER')
    card_table.setStyle(TableStyle([
        ('LEFTPADDING',   (0,0), (-1,-1), 3),
        ('RIGHTPADDING',  (0,0), (-1,-1), 3),
        ('TOPPADDING',    (0,0), (-1,-1), 0),
        ('BOTTOMPADDING', (0,0), (-1,-1), 6),
    ]))
    story.append(card_table)
    story.append(Spacer(1, 7*mm))

    # SECTION 2 — DATASET
    story.append(SectionHeader(2, 'Dataset Overview'))
    story.append(Spacer(1, 5*mm))
    story.append(Paragraph(
        "The training dataset was assembled from publicly available bone fracture "
        "X-ray repositories and filtered to retain only shoulder and arm fracture cases. "
        "Images underwent standard preprocessing including resizing and normalization "
        "before being split into training, validation, and test partitions.",
        S['body']))
    story.append(Spacer(1, 5*mm))

    ds_hdr = [Paragraph(f'<b>{h}</b>', S['label_bold'])
              for h in ['Split', 'Images', 'Annotations', '% of Total']]
    ds_rows = [
        ['Training',   f'{TRAIN_IMAGES:,}', f'{TRAIN_ANNOTS:,}', f'{TRAIN_IMAGES/TOTAL_IMAGES*100:.1f}%'],
        ['Validation', f'{VAL_IMAGES:,}',   f'{VAL_ANNOTS:,}',   f'{VAL_IMAGES/TOTAL_IMAGES*100:.1f}%'],
        ['Test',       f'{TEST_IMAGES:,}',  f'{TEST_ANNOTS:,}',  f'{TEST_IMAGES/TOTAL_IMAGES*100:.1f}%'],
        ['Total',      f'{TOTAL_IMAGES:,}', f'{TOTAL_ANNOTS:,}', '100%'],
    ]
    ds_data = [ds_hdr] + [[Paragraph(c, S['body_small']) for c in row] for row in ds_rows]
    ds_table = Table(ds_data, colWidths=[42*mm, 40*mm, 45*mm, 38*mm], hAlign='LEFT')
    ds_table.setStyle(TableStyle([
        ('BACKGROUND',    (0, 0), (-1,  0), RL_DARK_BLUE),
        ('TEXTCOLOR',     (0, 0), (-1,  0), RL_WHITE),
        ('BACKGROUND',    (0, 4), (-1,  4), RL_LIGHT_TEAL),
        ('FONTNAME',      (0, 4), (-1,  4), 'Helvetica-Bold'),
        ('ROWBACKGROUNDS',(0, 1), (-1, -2), [RL_LIGHT_BLUE, RL_WHITE, RL_LIGHT_BLUE]),
        ('ALIGN',         (0, 0), (-1, -1), 'CENTER'),
        ('VALIGN',        (0, 0), (-1, -1), 'MIDDLE'),
        ('GRID',          (0, 0), (-1, -1), 0.4, HexColor('#BDC3C7')),
        ('TOPPADDING',    (0, 0), (-1, -1), 6),
        ('BOTTOMPADDING', (0, 0), (-1, -1), 6),
    ]))
    story.append(ds_table)
    story.append(Spacer(1, 8*mm))

    # SECTION 3 — TRAINING PROGRESS
    story.append(SectionHeader(3, 'Training Progress'))
    story.append(Spacer(1, 5*mm))
    story.append(Paragraph(
        f"Training was conducted using YOLOv8m pretrained on COCO. The model was trained "
        f"for <b>{TOTAL_EPOCHS} epochs</b> on Kaggle with T4 x2 GPUs. The training "
        f"achieved a peak validation mAP50 of <b>{BEST_MAP50:.2f}% at epoch {BEST_EPOCH}</b>. "
        f"The model plateaued around epoch 50, maintaining high performance through epoch 86.",
        S['body']))
    story.append(Spacer(1, 5*mm))

    chart_buf = make_training_chart()
    story.append(Image(chart_buf, width=165*mm, height=68*mm))
    story.append(Spacer(1, 2*mm))
    story.append(Paragraph(
        'Figure 1 - YOLOv8m training progress (mAP50 over epochs). '
        'Gold star marks the peak. Orange dashed line shows test mAP50.', S['caption']))
    story.append(Spacer(1, 6*mm))

    # SECTION 4 — TEST RESULTS
    story.append(SectionHeader(4, 'Test Results'))
    story.append(Spacer(1, 5*mm))
    story.append(Paragraph(
        f"The final model (epoch {BEST_EPOCH} checkpoint) was evaluated on the "
        f"held-out test set of <b>{TEST_IMAGES:,} images</b> with <b>{TEST_ANNOTS:,} fracture instances</b>. "
        f"YOLOv8m achieved <b>{TEST_MAP50:.2f}% mAP50</b> at IoU=0.50, demonstrating excellent "
        f"generalization to unseen data with high precision ({TEST_PRECISION:.1f}%) and recall "
        f"({TEST_RECALL:.1f}%).",
        S['body']))
    story.append(Spacer(1, 6*mm))

    bar_buf = make_metrics_bar()
    story.append(Image(bar_buf, width=140*mm, height=60*mm))
    story.append(Paragraph('Figure 2 - YOLOv8m test set performance metrics.', S['caption']))
    story.append(Spacer(1, 6*mm))

    # SECTION 5 — COMPARISON
    story.append(SectionHeader(5, 'Comparison: Faster R-CNN vs YOLOv8'))
    story.append(Spacer(1, 5*mm))
    story.append(Paragraph(
        f"<b>YOLOv8m significantly outperforms the previous Faster R-CNN model</b>, achieving "
        f"a {TEST_MAP50 - 48.6:.1f}% absolute improvement in mAP50. The YOLOv8 model also provides "
        f"<b>perfect prediction consistency</b> (deterministic outputs) compared to the "
        f"inconsistent predictions from Faster R-CNN, and is <b>~10x faster</b> for inference.",
        S['body']))
    story.append(Spacer(1, 5*mm))

    comp_buf = make_comparison_chart()
    story.append(Image(comp_buf, width=140*mm, height=65*mm))
    story.append(Paragraph('Figure 3 - Performance comparison: Faster R-CNN vs YOLOv8m.', S['caption']))
    story.append(Spacer(1, 8*mm))

    # Comparison table
    comp_hdr = [Paragraph(f'<b>{h}</b>', S['label_bold']) for h in ['Metric', 'Faster R-CNN', 'YOLOv8m (Ours)']]
    comp_rows = [
        ['mAP50', '48.6%', f'{TEST_MAP50:.1f}%'],
        ['Precision', 'N/A', f'{TEST_PRECISION:.1f}%'],
        ['Recall', 'N/A', f'{TEST_RECALL:.1f}%'],
        ['Inference Speed', '~0.5s/image', '~0.03s/image'],
        ['Prediction Consistency', '❌ Inconsistent', '✅ Deterministic'],
        ['Training Time', '~24 hours', f'{TOTAL_HOURS}'],
    ]
    comp_data = [comp_hdr] + [[Paragraph(c, S['body_small']) for c in row] for row in comp_rows]
    comp_table = Table(comp_data, colWidths=[55*mm, 55*mm, 55*mm], hAlign='LEFT')
    comp_table.setStyle(TableStyle([
        ('BACKGROUND',    (0, 0), (-1,  0), RL_DARK_BLUE),
        ('TEXTCOLOR',     (0, 0), (-1,  0), RL_WHITE),
        ('ROWBACKGROUNDS',(0, 1), (-1, -1), [RL_WHITE, RL_LIGHT_BLUE]),
        ('ALIGN',         (0, 0), (-1, -1), 'CENTER'),
        ('VALIGN',        (0, 0), (-1, -1), 'MIDDLE'),
        ('GRID',          (0, 0), (-1, -1), 0.4, HexColor('#BDC3C7')),
        ('FONTNAME',      (0, 0), (-1,  0), 'Helvetica-Bold'),
        ('TOPPADDING',    (0, 0), (-1, -1), 6),
        ('BOTTOMPADDING', (0, 0), (-1, -1), 6),
    ]))
    story.append(comp_table)
    story.append(PageBreak())

    # SECTION 6 — CONCLUSION
    story.append(SectionHeader(6, 'Conclusion and Next Steps'))
    story.append(Spacer(1, 5*mm))
    story.append(Paragraph(
        f"The YOLOv8m model successfully achieved <b>{TEST_MAP50:.2f}% mAP50</b> on the test set, "
        f"demonstrating superior performance for shoulder and arm fracture detection. Compared to "
        f"the previous Faster R-CNN model, YOLOv8 provides a <b>{(TEST_MAP50 - 48.6):.1f}% improvement</b> "
        f"in detection accuracy with perfect prediction consistency. The model is ready for deployment "
        f"in the <b>AETHEA medical platform</b>.",
        S['body']))
    story.append(Spacer(1, 5*mm))

    next_steps = [
        ('Deploy YOLOv8 to AETHEA Backend',
         'Integrate the best.pt checkpoint into the FastAPI inference service for real-time X-ray analysis.'),
        ('Export to ONNX/TensorRT',
         'Optimize inference speed by exporting to ONNX or TensorRT format for production deployment.'),
        ('Expand Egyptian Population Data',
         'Collect additional local X-rays to further improve generalization on target demographics.'),
        ('Continuous Learning Loop',
         'Implement radiologist feedback capture to enable periodic model fine-tuning on corrected predictions.'),
        ('Add Fracture Classification',
         'Extend model to classify fracture types (e.g., simple, comminuted, displaced) as future work.'),
    ]
    for title, desc in next_steps:
        step_table = Table(
            [[Paragraph(f'<b>{title}</b>', S['highlight']),
              Paragraph(desc, S['body_small'])]],
            colWidths=[52*mm, 113*mm])
        step_table.setStyle(TableStyle([
            ('BACKGROUND',    (0, 0), (0, 0), RL_LIGHT_TEAL),
            ('BACKGROUND',    (1, 0), (1, 0), RL_WHITE),
            ('VALIGN',        (0, 0), (-1,-1), 'TOP'),
            ('LEFTPADDING',   (0, 0), (-1,-1), 7),
            ('TOPPADDING',    (0, 0), (-1,-1), 6),
            ('BOTTOMPADDING', (0, 0), (-1,-1), 6),
            ('GRID',          (0, 0), (-1,-1), 0.4, HexColor('#BDC3C7')),
        ]))
        story.append(step_table)
        story.append(Spacer(1, 2*mm))

    story.append(Spacer(1, 8*mm))
    story.append(HRFlowable(width='100%', thickness=0.8, color=RL_TEAL))
    story.append(Spacer(1, 4*mm))
    story.append(Paragraph(
        f'{AUTHOR_NAME}  |  {PROJECT_NAME}  |  {MODEL_NAME}', S['sig']))
    story.append(Paragraph(
        f'Report generated on {datetime.now().strftime("%B %d, %Y at %H:%M")}', S['sig']))

    # BUILD
    print("📊 Generating charts...")
    doc.build(story, onFirstPage=first_page, onLaterPages=normal_page)
    print(f"\n✅ PDF saved to:\n   {OUTPUT_PDF}")
    print("=" * 60)


# ============================================================================
# ENTRY POINT
# ============================================================================

if __name__ == '__main__':
    build_pdf()

  AETHEA — YOLOv8 Final Report Generator
📊 Generating charts...

✅ PDF saved to:
   C:\Users\andre\Documents\Graduation Project\Shoulders\100 epoch YOLO\RUN 1\YOLOv8_FINAL_REPORT.pdf
